# Jaccpot GPU N-Ladder Production Sweep

This notebook benchmarks a fixed `engblom` production regime across a particle-count ladder.

Goals:
- identify the smallest stable traversal seed per `N`
- compare total steady-state runtime across a small candidate grid
- emit a compact recommendation table for production runs


In [ ]:
import os

USE_AUTOCVD = False
AUTOCVD_NUM_GPUS = 1
AUTOCVD_LEAST_USED = True
AUTOCVD_EXCLUDE = []
MANUAL_CUDA_VISIBLE_DEVICES = None

if MANUAL_CUDA_VISIBLE_DEVICES is not None:
    os.environ["CUDA_VISIBLE_DEVICES"] = MANUAL_CUDA_VISIBLE_DEVICES
    print("Set CUDA_VISIBLE_DEVICES =", os.environ["CUDA_VISIBLE_DEVICES"])
elif USE_AUTOCVD:
    try:
        from autocvd import autocvd

        autocvd(
            num_gpus=AUTOCVD_NUM_GPUS,
            least_used=AUTOCVD_LEAST_USED,
            exclude=AUTOCVD_EXCLUDE,
        )
        print("autocvd selected CUDA_VISIBLE_DEVICES =", os.environ.get("CUDA_VISIBLE_DEVICES"))
    except ImportError:
        print("autocvd is not installed. Using existing CUDA visibility.")
else:
    print("Using existing CUDA visibility:", os.environ.get("CUDA_VISIBLE_DEVICES", "<all visible>"))

INDEX_PRECISION = "int32"
os.environ.setdefault("JACCPOT_INDEX_PRECISION", INDEX_PRECISION)
os.environ.setdefault("YGGDRAX_INDEX_PRECISION", INDEX_PRECISION)
os.environ.setdefault("TF_GPU_ALLOCATOR", "cuda_malloc_async")
os.environ.setdefault("XLA_PYTHON_CLIENT_PREALLOCATE", "false")
os.environ.setdefault("XLA_PYTHON_CLIENT_ALLOCATOR", "platform")

if "--xla_gpu_enable_command_buffer=" not in os.environ.get("XLA_FLAGS", ""):
    existing_xla_flags = os.environ.get("XLA_FLAGS", "").strip()
    command_buffer_off = "--xla_gpu_enable_command_buffer="
    os.environ["XLA_FLAGS"] = (
        f"{existing_xla_flags} {command_buffer_off}".strip()
        if existing_xla_flags
        else command_buffer_off
    )

VISIBLE_PHYSICAL_GPUS = [
    part.strip()
    for part in os.environ.get("CUDA_VISIBLE_DEVICES", "").split(",")
    if part.strip() != ""
]
NVIDIA_SMI_GPU_INDEX = int(VISIBLE_PHYSICAL_GPUS[0]) if VISIBLE_PHYSICAL_GPUS else 0
os.environ["JACCPOT_NVIDIA_SMI_GPU_INDEX"] = str(int(NVIDIA_SMI_GPU_INDEX))

print("CUDA_VISIBLE_DEVICES:", os.environ.get("CUDA_VISIBLE_DEVICES", "<all visible>"))
print("JACCPOT_INDEX_PRECISION:", os.environ.get("JACCPOT_INDEX_PRECISION"))
print("nvidia-smi physical GPU index:", NVIDIA_SMI_GPU_INDEX)


In [ ]:
import gc
import json
import pathlib
import subprocess
import sys
import time
from dataclasses import replace

import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

REPO_ROOT = pathlib.Path.cwd().resolve()
if not (REPO_ROOT / "jaccpot").exists():
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) in sys.path:
    sys.path.remove(str(REPO_ROOT))
sys.path.insert(0, str(REPO_ROOT))

from jaccpot import (
    FMMAdvancedConfig,
    FMMPreset,
    FarFieldConfig,
    FastMultipoleMethod,
    NearFieldConfig,
    RuntimePolicyConfig,
    TreeConfig,
)
from yggdrax.interactions import DualTreeTraversalConfig

all_devices = jax.devices()
gpu_devices = [d for d in all_devices if d.platform == "gpu"]
print("JAX backend:", jax.default_backend())
print("Visible devices:", all_devices)
if not gpu_devices:
    raise RuntimeError("No GPU visible to JAX.")
print("Using GPU:", gpu_devices[0])


In [ ]:
# Production-oriented ladder configuration.
particle_counts = [65536, 131072, 262144, 524288]
leaf_size = 128
max_order = 4
softening = 1e-3
working_dtype = jnp.float32
seed = 0
runs = 1
warmup = 1
benchmark_scope = "steady_eval"
peak_poll_interval_s = 0.02
baseline_profile = "engblom_n_ladder_production"

traversal_candidates = [
    {"max_pair_queue": 131072, "process_block": 128, "max_interactions_per_node": 8192, "max_neighbors_per_leaf": 2048},
    {"max_pair_queue": 262144, "process_block": 256, "max_interactions_per_node": 8192, "max_neighbors_per_leaf": 4096},
    {"max_pair_queue": 262144, "process_block": 256, "max_interactions_per_node": 16384, "max_neighbors_per_leaf": 4096},
    {"max_pair_queue": 524288, "process_block": 256, "max_interactions_per_node": 16384, "max_neighbors_per_leaf": 4096},
]
m2l_chunk_candidates = [1024, 512]
nearfield_chunk_candidates = [128, 64]

output_dir = REPO_ROOT / "benchmarks" / "n_ladder_production"
output_dir.mkdir(parents=True, exist_ok=True)

base_advanced = FMMAdvancedConfig(
    tree=TreeConfig(
        tree_type="radix",
        mode="lbvh",
        leaf_target=64,
        refine_local=False,
        max_refine_levels=0,
        aspect_threshold=16.0,
    ),
    farfield=FarFieldConfig(
        grouped_interactions=False,
        mode="pair_grouped",
        rotation="solidfmm",
        m2l_chunk_size=1024,
        l2l_chunk_size=None,
        streamed_far_pairs=True,
        mixed_order=False,
        mixed_order_min_order=None,
    ),
    nearfield=NearFieldConfig(
        mode="bucketed",
        edge_chunk_size=128,
        precompute_scatter_schedules=False,
    ),
    runtime=RuntimePolicyConfig(
        host_refine_mode="off",
        fail_fast=True,
        jit_tree=True,
        jit_traversal=True,
        memory_objective="minimum_memory",
        traversal_config=DualTreeTraversalConfig(
            max_pair_queue=262144,
            process_block=256,
            max_interactions_per_node=8192,
            max_neighbors_per_leaf=4096,
        ),
        pair_process_block=None,
        enable_interaction_cache=False,
        retain_traversal_result=False,
        retain_interactions=False,
        autotune_m2l_chunk=False,
        upward_leaf_batch_size=2048,
    ),
    mac_type="engblom",
    dehnen_radius_scale=1.0,
)

base_fmm_kwargs = dict(
    preset=FMMPreset.LARGE_N_GPU,
    basis="solidfmm",
    precision="fp32",
    theta=0.6,
    softening=softening,
    working_dtype=working_dtype,
    adaptive_order=False,
    advanced=base_advanced,
)

candidate_rows = []
for traversal_cfg in traversal_candidates:
    for m2l_chunk_size in m2l_chunk_candidates:
        for nearfield_chunk_size in nearfield_chunk_candidates:
            candidate_rows.append(
                {
                    "max_pair_queue": int(traversal_cfg["max_pair_queue"]),
                    "process_block": int(traversal_cfg["process_block"]),
                    "max_interactions_per_node": int(traversal_cfg["max_interactions_per_node"]),
                    "max_neighbors_per_leaf": int(traversal_cfg["max_neighbors_per_leaf"]),
                    "m2l_chunk_size": int(m2l_chunk_size),
                    "nearfield_edge_chunk_size": int(nearfield_chunk_size),
                }
            )
candidate_df = pd.DataFrame(candidate_rows)
config_df = pd.DataFrame(
    [
        {
            "baseline_profile": baseline_profile,
            "particle_counts": ",".join(str(v) for v in particle_counts),
            "leaf_size": int(leaf_size),
            "max_order": int(max_order),
            "runs": int(runs),
            "warmup": int(warmup),
            "benchmark_scope": benchmark_scope,
            "working_dtype": str(jnp.dtype(working_dtype)),
        }
    ]
)
display(config_df)
display(candidate_df)


In [ ]:
def classify_worker_error(message):
    text = str(message).lower()
    if text.strip() == "":
        return ""
    if "pair queue capacity exceeded" in text or "interaction capacity exceeded" in text:
        return "interaction_capacity"
    if "neighbor list capacity exceeded" in text:
        return "neighbor_capacity"
    if "resource_exhausted" in text or "out of memory" in text:
        return "oom"
    return "other_error"


def _query_gpu_memory_mb_by_pid(pid):
    cmd = [
        "nvidia-smi",
        f"--id={int(NVIDIA_SMI_GPU_INDEX)}",
        "--query-compute-apps=pid,used_memory",
        "--format=csv,noheader,nounits",
    ]
    out = subprocess.run(cmd, check=True, capture_output=True, text=True)
    total_used_mb = 0.0
    for line in out.stdout.strip().splitlines():
        parts = [part.strip() for part in line.split(",")]
        if len(parts) < 2:
            continue
        try:
            row_pid = int(parts[0])
            row_used = float(parts[1])
        except Exception:
            continue
        if row_pid == int(pid):
            total_used_mb += float(row_used)
    return float(total_used_mb)


def serialize_fmm_kwargs_for_worker(fmm_kwargs):
    probe_fmm = FastMultipoleMethod(**fmm_kwargs)
    try:
        advanced = probe_fmm.advanced
        traversal_cfg = advanced.runtime.traversal_config
        traversal_payload = None
        if traversal_cfg is not None:
            traversal_payload = {
                "process_block": int(traversal_cfg.process_block),
                "max_neighbors_per_leaf": int(traversal_cfg.max_neighbors_per_leaf),
                "max_interactions_per_node": int(traversal_cfg.max_interactions_per_node),
                "max_pair_queue": int(traversal_cfg.max_pair_queue),
            }
        payload = {
            "preset": "large_n_gpu",
            "basis": str(fmm_kwargs.get("basis", "solidfmm")),
            "theta": float(fmm_kwargs.get("theta", 0.6)),
            "softening": float(fmm_kwargs.get("softening", 1e-3)),
            "working_dtype": str(jnp.dtype(getattr(probe_fmm._impl, "working_dtype", jnp.float32))),
            "tree_type": str(advanced.tree.tree_type),
            "leaf_target": int(advanced.tree.leaf_target),
            "farfield_rotation": str(advanced.farfield.rotation),
            "farfield_mode": str(advanced.farfield.mode),
            "grouped_interactions": bool(advanced.farfield.grouped_interactions),
            "streamed_far_pairs": advanced.farfield.streamed_far_pairs,
            "mixed_order": bool(advanced.farfield.mixed_order),
            "mixed_order_min_order": advanced.farfield.mixed_order_min_order,
            "nearfield_mode": str(advanced.nearfield.mode),
            "nearfield_edge_chunk_size": int(advanced.nearfield.edge_chunk_size),
            "precompute_scatter_schedules": bool(advanced.nearfield.precompute_scatter_schedules),
            "pair_process_block": (
                None if advanced.runtime.pair_process_block is None else int(advanced.runtime.pair_process_block)
            ),
            "memory_objective": str(advanced.runtime.memory_objective),
            "fail_fast": bool(advanced.runtime.fail_fast),
            "jit_traversal": bool(advanced.runtime.jit_traversal),
            "traversal_config": traversal_payload,
            "enable_interaction_cache": bool(advanced.runtime.enable_interaction_cache),
            "retain_traversal_result": bool(advanced.runtime.retain_traversal_result),
            "retain_interactions": bool(advanced.runtime.retain_interactions),
            "autotune_m2l_chunk": bool(advanced.runtime.autotune_m2l_chunk),
            "adaptive_order": bool(getattr(probe_fmm._impl, "adaptive_order", False)),
            "p_gears": [int(v) for v in getattr(probe_fmm._impl, "p_gears", tuple())],
            "adaptive_error_model": str(getattr(probe_fmm._impl, "adaptive_error_model", "tail_proxy")),
            "adaptive_eps": (
                None if getattr(probe_fmm._impl, "adaptive_eps", None) is None else float(getattr(probe_fmm._impl, "adaptive_eps"))
            ),
            "mac_force_scale_mode": str(getattr(probe_fmm._impl, "mac_force_scale_mode", "prev")),
            "mac_type": str(advanced.mac_type),
            "benchmark_scope": str(benchmark_scope),
            "worker_autotune_traversal": False,
            "worker_autotune_nearfield_chunk": False,
            "traversal_candidates": [],
            "nearfield_chunk_candidates": [],
        }
        return payload
    finally:
        clear_fn = getattr(probe_fmm, "clear_runtime_caches", None)
        if callable(clear_fn):
            clear_fn(clear_jax_compilation=True)
        jax.clear_caches()
        gc.collect()
        del probe_fmm


def with_runtime_knobs(base_fmm_kwargs, *, traversal_cfg, m2l_chunk_size, nearfield_edge_chunk_size):
    advanced = base_fmm_kwargs["advanced"]
    runtime_cfg = replace(
        advanced.runtime,
        traversal_config=DualTreeTraversalConfig(
            max_pair_queue=int(traversal_cfg["max_pair_queue"]),
            process_block=int(traversal_cfg["process_block"]),
            max_interactions_per_node=int(traversal_cfg["max_interactions_per_node"]),
            max_neighbors_per_leaf=int(traversal_cfg["max_neighbors_per_leaf"]),
        ),
    )
    farfield_cfg = replace(advanced.farfield, m2l_chunk_size=int(m2l_chunk_size))
    nearfield_cfg = replace(advanced.nearfield, edge_chunk_size=int(nearfield_edge_chunk_size))
    out = dict(base_fmm_kwargs)
    out["advanced"] = replace(advanced, runtime=runtime_cfg, farfield=farfield_cfg, nearfield=nearfield_cfg)
    return out


def run_worker_case(num_particles, *, traversal_cfg, m2l_chunk_size, nearfield_edge_chunk_size):
    worker_script = REPO_ROOT / "examples" / "benchmark_gpu_radix_worker.py"
    runtime_kwargs = with_runtime_knobs(
        base_fmm_kwargs,
        traversal_cfg=traversal_cfg,
        m2l_chunk_size=m2l_chunk_size,
        nearfield_edge_chunk_size=nearfield_edge_chunk_size,
    )
    payload = serialize_fmm_kwargs_for_worker(runtime_kwargs)
    ready_marker = "__JACCPOT_WORKER_READY__"
    cmd = [
        sys.executable,
        str(worker_script),
        "--mode",
        "sweep",
        "--num-particles",
        str(int(num_particles)),
        "--leaf-size",
        str(int(leaf_size)),
        "--max-order",
        str(int(max_order)),
        "--runs",
        str(int(runs)),
        "--warmup",
        str(int(warmup)),
        "--dtype",
        str(jnp.dtype(working_dtype)),
        "--seed",
        str(int(seed)),
        "--emit-ready-marker",
        "--config-json",
        json.dumps(payload),
    ]
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True, bufsize=1)
    stdout_lines = []
    ready_seen = False
    while proc.poll() is None:
        line = proc.stdout.readline()
        if not line:
            continue
        line = line.strip()
        if not line:
            continue
        if line == ready_marker:
            ready_seen = True
            break
        stdout_lines.append(line)
    samples = []
    baseline_used_mb = float(_query_gpu_memory_mb_by_pid(proc.pid)) if ready_seen else 0.0
    while proc.poll() is None:
        ts = time.perf_counter()
        try:
            used_mb = _query_gpu_memory_mb_by_pid(proc.pid)
            samples.append({"t_s": ts, "gpu_used_mb": used_mb})
        except Exception:
            pass
        time.sleep(float(peak_poll_interval_s))
    stdout, stderr = proc.communicate()
    if stdout:
        stdout_lines.extend(line.strip() for line in stdout.splitlines() if line.strip())
    trace_df = pd.DataFrame(samples)
    peak_used_mb = float(trace_df["gpu_used_mb"].max()) if not trace_df.empty else float(baseline_used_mb)
    row = {
        "num_particles": int(num_particles),
        "max_pair_queue": int(traversal_cfg["max_pair_queue"]),
        "process_block": int(traversal_cfg["process_block"]),
        "max_interactions_per_node": int(traversal_cfg["max_interactions_per_node"]),
        "max_neighbors_per_leaf": int(traversal_cfg["max_neighbors_per_leaf"]),
        "m2l_chunk_size": int(m2l_chunk_size),
        "nearfield_edge_chunk_size": int(nearfield_edge_chunk_size),
        "gpu_peak_delta_mb": float(max(0.0, peak_used_mb - baseline_used_mb)),
        "fit_status": "ok",
        "error": "",
    }
    if proc.returncode != 0:
        msg = (stderr or "\n".join(stdout_lines) or "").strip()
        row.update({"fit_status": classify_worker_error(msg), "error": msg})
        return row, trace_df
    lines = [line for line in stdout_lines if line != ready_marker]
    if not lines:
        row.update({"fit_status": "other_error", "error": "worker produced no output"})
        return row, trace_df
    payload_out = json.loads(lines[-1])
    row.update(payload_out)
    row["fit_status"] = "ok" if str(payload_out.get("error", "")) == "" else classify_worker_error(payload_out.get("error", ""))
    return row, trace_df


def capacity_rank_key(df):
    return df.sort_values(
        [
            "max_pair_queue",
            "max_interactions_per_node",
            "max_neighbors_per_leaf",
            "process_block",
            "prepare_mean_seconds",
            "evaluate_mean_seconds",
        ],
        ascending=[True, True, True, True, True, True],
    )


In [ ]:
ladder_run_id = time.strftime("%Y%m%d_%H%M%S")
print(f"Starting N-ladder production sweep run_id={ladder_run_id}")
sweep_rows = []
for num_particles in particle_counts:
    print(f"Running N={int(num_particles)}")
    for traversal_cfg in traversal_candidates:
        for m2l_chunk_size in m2l_chunk_candidates:
            for nearfield_edge_chunk_size in nearfield_chunk_candidates:
                row, _ = run_worker_case(
                    int(num_particles),
                    traversal_cfg=traversal_cfg,
                    m2l_chunk_size=int(m2l_chunk_size),
                    nearfield_edge_chunk_size=int(nearfield_edge_chunk_size),
                )
                row["run_id"] = str(ladder_run_id)
                sweep_rows.append(row)
ladder_df = pd.DataFrame(sweep_rows)
ladder_df.to_csv(output_dir / f"n_ladder_production_{ladder_run_id}.csv", index=False)
ladder_df.to_csv(output_dir / "n_ladder_production_latest.csv", index=False)
ladder_df


In [ ]:
if ladder_df.empty:
    print("No ladder rows produced.")
else:
    stable_df = ladder_df[ladder_df["fit_status"] == "ok"].copy()
    fail_df = ladder_df[ladder_df["fit_status"] != "ok"].copy()

    print("=== Fastest Stable Configurations By N ===")
    if stable_df.empty:
        print("No stable configurations found.")
    else:
        fastest_df = (
            stable_df.assign(total_phase_seconds=stable_df["prepare_mean_seconds"] + stable_df["evaluate_mean_seconds"])
            .sort_values(["num_particles", "total_phase_seconds", "prepare_mean_seconds"], ascending=[True, True, True])
            .groupby("num_particles", as_index=False)
            .first()
        )
        display(
            fastest_df[[
                "num_particles",
                "total_phase_seconds",
                "prepare_mean_seconds",
                "evaluate_mean_seconds",
                "max_pair_queue",
                "process_block",
                "max_interactions_per_node",
                "max_neighbors_per_leaf",
                "m2l_chunk_size",
                "nearfield_edge_chunk_size",
            ]]
        )

        print("\n=== Smallest Stable Capacity Configurations By N ===")
        recommended_df = (
            capacity_rank_key(stable_df)
            .groupby("num_particles", as_index=False)
            .first()
        )
        recommended_df = recommended_df.assign(
            total_phase_seconds=recommended_df["prepare_mean_seconds"] + recommended_df["evaluate_mean_seconds"]
        )
        display(
            recommended_df[[
                "num_particles",
                "total_phase_seconds",
                "prepare_mean_seconds",
                "evaluate_mean_seconds",
                "gpu_peak_delta_mb",
                "max_pair_queue",
                "process_block",
                "max_interactions_per_node",
                "max_neighbors_per_leaf",
                "m2l_chunk_size",
                "nearfield_edge_chunk_size",
            ]]
        )

        print("\n=== Timing Trend (Recommended Capacity) ===")
        fig, ax = plt.subplots(figsize=(7, 4))
        ax.plot(recommended_df["num_particles"], recommended_df["prepare_mean_seconds"], marker="o", label="prepare")
        ax.plot(recommended_df["num_particles"], recommended_df["evaluate_mean_seconds"], marker="s", label="evaluate")
        ax.plot(recommended_df["num_particles"], recommended_df["total_phase_seconds"], marker="^", label="total")
        ax.set_xscale("log", base=2)
        ax.set_yscale("log")
        ax.set_xlabel("num_particles")
        ax.set_ylabel("seconds")
        ax.legend()
        ax.set_title("Production Ladder Timing")
        plt.show()

    print("\n=== Failures ===")
    if fail_df.empty:
        print("No failures.")
    else:
        display(
            fail_df[[
                "num_particles",
                "fit_status",
                "max_pair_queue",
                "process_block",
                "max_interactions_per_node",
                "max_neighbors_per_leaf",
                "m2l_chunk_size",
                "nearfield_edge_chunk_size",
                "error",
            ]]
        )
